# 02 – PU Assist: multimodal RAG chatbot

**Before this:** run `01_model_setup.ipynb` once (libraries + model download).
Keep `pu_corpus.jsonl.gz` and `campus_map.png` in the same folder as this notebook.

How a question is answered:

```
student question (typed or voice note, + optional image)
   │
   ├─ voice note? → Whisper turns it into text (English / Hindi / Kannada / …)
   ├─ Qwen3-VL rewrites it in the documents' wording (+ chat history for follow-ups)
   ├─ image?  → Qwen3-VL reads it (transcribes notice/timetable) and suggests another query
   ▼
Qwen3-Embedding → FAISS (166k chunks, top 50 per query) → bge-reranker → best 6 passages
   ▼
Qwen3-VL answers from those passages (+ the image, + campus map for "where is…") with [1] [2] citations
```

| Part | Steps | First run | Later runs |
|---|---|---|---|
| A. Search index | 1–5 | ~8 min (embeds 166k chunks once, ~350 chunks/s on the H200) | < 1 min (loads saved index) |
| B. Chatbot | 6–8 | ~1–2 min (loads Qwen3-VL + Whisper) | ~1–2 min |
| C. Try it | 9–15 | ~10–40 s per question | |
| D. Chat box in the notebook | 16 | | |
| E. Gradio web page (optional) | 17 | | |

---
# A. Search index
## 1. Settings

In [1]:
import os, time, torch

DATA_DIR   = "."                                   # folder with pu_corpus.jsonl.gz and campus_map.png
CORPUS     = os.path.join(DATA_DIR, "pu_corpus.jsonl.gz")
CAMPUS_MAP = os.path.join(DATA_DIR, "campus_map.png")
INDEX_DIR  = os.path.join(DATA_DIR, "rag_index")    # saved FAISS index goes here
MODEL_ID   = "Qwen/Qwen3-VL-30B-A3B-Instruct"       # <- change the vision-language model here (see README)
MODEL_DIR  = os.path.expanduser(f"~/hf_models/{MODEL_ID.split('/')[-1]}")

EMBED_MODEL  = "Qwen/Qwen3-Embedding-0.6B"          # 1024-dim, multilingual
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"
ASR_MODEL    = "openai/whisper-large-v3-turbo"     # speech-to-text for voice questions

CANDIDATES     = 50      # passages pulled from FAISS per search query (2-3 queries per question)
TOP_K          = 6       # passages given to Qwen3-VL after reranking
MAX_PER_SOURCE = 3       # at most this many chunks from the same page/PDF
MAX_UNOFFICIAL = 3       # at most this many student photos/videos/interview/tour chunks
CONTEXT_CHARS  = 300     # each chunk is indexed with the end of the chunk before it (keeps headings)
REBUILD_INDEX  = False   # set True after uploading a new pu_corpus.jsonl.gz

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

Device: cuda | NVIDIA H200 NVL


## 2. Load the corpus

In [2]:
import gzip, json, collections

with gzip.open(CORPUS, "rt", encoding="utf-8") as f:
    corpus = [json.loads(line) for line in f]

print(f"{len(corpus):,} chunks")
for cls, n in collections.Counter(r["source_class"] for r in corpus).most_common():
    print(f"  {cls:30s} {n:>8,}")

166,180 chunks
  official_document               140,273
  repository_metadata              14,145
  official_website                 10,610
  official_program_regulation         660
  user_video                          160
  campus_tour                         139
  user_photo                           80
  user_interview                       68
  library_catalog                      26
  external                              8
  campus_map                            7
  teacher_audio                         4


Chunks were cut every ~1000 characters, so a heading like *"8.9 Minimum Performance Criteria – Theory courses"*
can end up in one chunk and the rules in the next. Each passage below therefore starts with the last
`CONTEXT_CHARS` characters of the chunk before it (same document). These `passages` are what gets
embedded, reranked and shown to Qwen3-VL.

In [3]:
import re

def _split_id(chunk_id):                      # "doc-abc-7" -> ("doc-abc", 7)
    m = re.match(r"(.+)-(\d+)$", chunk_id)
    return (m.group(1), int(m.group(2))) if m else (None, None)

pos_by_id, last_chunk = {}, {}                 # last_chunk: base -> (n, id) of its final chunk
for i, r in enumerate(corpus):
    pos_by_id[r["id"]] = i
    base, n = _split_id(r["id"])
    if base and n >= last_chunk.get(base, (-1, None))[0]:
        last_chunk[base] = (n, r["id"])

def previous_chunk_id(chunk_id):
    """doc-abc-7 -> doc-abc-6;  paged ids like userpdf-12-0 -> last chunk of page 11."""
    base, n = _split_id(chunk_id)
    if base is None:
        return None
    if n > 0:
        return f"{base}-{n - 1}"
    doc, page = _split_id(base)
    return last_chunk.get(f"{doc}-{page - 1}", (None, None))[1] if doc else None

passages = []
for r in corpus:
    prev = pos_by_id.get(previous_chunk_id(r["id"]))
    tail = f"…{corpus[prev]['text'][-CONTEXT_CHARS:]} " if prev is not None else ""
    passages.append(f"{r.get('title', '')}\n{tail}{r['text']}".strip())

print(f"{sum(p is not None for p in (pos_by_id.get(previous_chunk_id(r['id'])) for r in corpus)):,} "
      f"of {len(corpus):,} passages carry context from the previous chunk")

145,437 of 166,180 passages carry context from the previous chunk


## 3. Load the search models

- **Qwen3-Embedding-0.6B** turns text into vectors (questions get an instruction prefix, passages don't).
- **bge-reranker-v2-m3** re-scores the top 40 hits by reading question + passage together — much more accurate than vectors alone.

In [4]:
from sentence_transformers import SentenceTransformer, CrossEncoder

t = time.time()
embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)
embedder.max_seq_length = 512          # chunks are ~250-300 tokens; caps the rare long one
if DEVICE == "cuda":
    embedder = embedder.to(torch.bfloat16)
reranker = CrossEncoder(RERANK_MODEL, max_length=512, device=DEVICE)
print(f"Search models loaded in {time.time() - t:.0f}s")

QUERY_PROMPT = ("Instruct: Given a student's question about Presidency University, "
                "retrieve passages from university documents that answer it\nQuery:")

def embed(texts, prompt=None, batch_size=128):
    with torch.inference_mode():
        v = embedder.encode(texts, prompt=prompt, batch_size=batch_size,
                            convert_to_tensor=True, show_progress_bar=False)
    return torch.nn.functional.normalize(v.float(), dim=-1).cpu().numpy()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Search models loaded in 35s


## 4. Build the FAISS index (first time ~8 min) or load the saved one

Vectors are normalized, so inner product = cosine similarity (`IndexFlatIP`, exact search).
The index is saved to `rag_index/` and reused next time.

In [5]:
import faiss, numpy as np

index_file = os.path.join(INDEX_DIR, "faiss.index")
info_file  = os.path.join(INDEX_DIR, "info.json")
info_now   = {"count": len(corpus), "first_id": corpus[0]["id"], "last_id": corpus[-1]["id"],
              "embed_model": EMBED_MODEL, "passages": f"title+prev{CONTEXT_CHARS}"}   # change -> auto rebuild

saved_ok = (os.path.exists(index_file) and os.path.exists(info_file)
            and json.load(open(info_file)) == info_now)

if saved_ok and not REBUILD_INDEX:
    index = faiss.read_index(index_file)
    print(f"Loaded saved index: {index.ntotal:,} vectors")
else:
    os.makedirs(INDEX_DIR, exist_ok=True)
    texts, index, BLOCK = passages, None, 10_000
    t = time.time()
    for start in range(0, len(texts), BLOCK):
        vecs = embed(texts[start:start + BLOCK])
        if index is None:
            index = faiss.IndexFlatIP(vecs.shape[1])
        index.add(vecs)
        done = start + len(vecs)
        rate = done / (time.time() - t)
        print(f"  {done:>7,}/{len(texts):,} | {rate:,.0f} chunks/s | ~{(len(texts) - done) / rate / 60:.1f} min left")
    faiss.write_index(index, index_file)
    json.dump(info_now, open(info_file, "w"))
    print(f"Built and saved index in {(time.time() - t) / 60:.1f} min")

   10,000/166,180 | 309 chunks/s | ~8.4 min left
   20,000/166,180 | 312 chunks/s | ~7.8 min left
   30,000/166,180 | 311 chunks/s | ~7.3 min left
   40,000/166,180 | 311 chunks/s | ~6.8 min left
   50,000/166,180 | 311 chunks/s | ~6.2 min left
   60,000/166,180 | 312 chunks/s | ~5.7 min left
   70,000/166,180 | 311 chunks/s | ~5.2 min left
   80,000/166,180 | 311 chunks/s | ~4.6 min left
   90,000/166,180 | 311 chunks/s | ~4.1 min left
  100,000/166,180 | 311 chunks/s | ~3.5 min left
  110,000/166,180 | 311 chunks/s | ~3.0 min left
  120,000/166,180 | 312 chunks/s | ~2.5 min left
  130,000/166,180 | 312 chunks/s | ~1.9 min left
  140,000/166,180 | 312 chunks/s | ~1.4 min left
  150,000/166,180 | 315 chunks/s | ~0.9 min left
  160,000/166,180 | 323 chunks/s | ~0.3 min left
  166,180/166,180 | 328 chunks/s | ~0.0 min left
Built and saved index in 8.5 min


## 5. Search + check retrieval quality (before loading the big model)

In [6]:
CLASS_LABEL = {
    "official_website":            "Official website",
    "official_document":           "Official document",
    "official_program_regulation": "Official programme regulation (B.Tech CSE 2023-27 only)",
    "repository_metadata":         "University repository listing",
    "library_catalog":             "Library catalogue",
    "campus_map":                  "Campus map",
    "campus_tour":                 "Virtual campus tour",
    "user_photo":                  "Student photo caption (unofficial)",
    "user_video":                  "Student campus video (unofficial, auto-transcribed)",
    "user_interview":              "Student interview (unofficial, auto-transcribed)",
    "teacher_audio":               "Teacher recording (auto-transcribed, not an official document)",
    "external":                    "External site (not university policy)",
}

def source_link(r):
    return r.get("url") or r.get("page_url") or r.get("local_path") or ""

def source_label(r):
    label = r.get("title") or CLASS_LABEL.get(r["source_class"], r["source_class"])
    return label + (f", page {r['page']}" if r.get("page") else "")

UNOFFICIAL = {"user_photo", "user_video", "user_interview", "teacher_audio", "campus_tour"}

def search(query, k=TOP_K, extra_queries=(), candidates=CANDIDATES):
    """Vector-search with the question (+ any rewritten queries), then rerank everything against `query`."""
    queries = [query] + [q for q in extra_queries if q and q.strip()]
    _, ids = index.search(embed(queries, prompt=QUERY_PROMPT), candidates)
    cand = list(dict.fromkeys(int(i) for row in ids for i in row if i >= 0))    # union, no repeats
    scores = reranker.predict([(query, passages[i][:2000]) for i in cand],
                              batch_size=32, show_progress_bar=False)
    results, per_source, unofficial = [], collections.Counter(), 0
    for j in np.argsort(-np.asarray(scores)):
        r = corpus[cand[j]]
        src = r.get("url") or r.get("title")
        if per_source[src] >= MAX_PER_SOURCE or (r["source_class"] in UNOFFICIAL and unofficial >= MAX_UNOFFICIAL):
            continue
        per_source[src] += 1
        unofficial += r["source_class"] in UNOFFICIAL
        results.append({**r, "passage": passages[cand[j]], "score": float(scores[j])})
        if len(results) == k:
            break
    return results

def show_hits(hits, width=150):
    for n, r in enumerate(hits, 1):
        print(f"[{n}] {r['score']:.2f} | {CLASS_LABEL.get(r['source_class'], r['source_class'])} | {source_label(r)[:70]}")
        print(f"     {source_link(r)}")
        print(f"     {r['text'][:width].replace(chr(10), ' ')}...")

show_hits(search("What is the minimum attendance needed to write the end-term exam?"))

[1] 0.51 | Official website | Presidency University
     https://presidencyuniversity.in/school/school-of-management/circulars
     Circular (PU202383) - Mandatory Attendance criterion of 75% minimum in order to be eligible to write the Examinations - AY 2023-24 CIRCULAR (PU2023090...
[2] 0.49 | Official document | PU Factsheet 2025-26
     https://presidencyuniversity.in/uploads/images/6791f7156eec81737619221.pdf
     urses are open to Incoming Exchange students. However, some have prerequisites or require permission from the relevant faculty or school before you ca...
[3] 0.34 | Official website | Presidency University
     https://presidencyuniversity.in/school/school-of-management/circulars
     Circular (PU202381) Course Registration for Higher Semester Students of all Programs for Odd Semester of the Academic Year 2023-24 (Except MBA) Circul...
[4] 0.27 | Official document | Examination Regulations, 2019
     https://presidencyuniversity.in/assets/PDF/Examination-Regulations-2019

In [7]:
# Retrieval test: does the right source show up in the top results?
EVAL = [
    ("What is the minimum attendance needed to write the end-term exam?",             "68903c797f1721754283129"),
    ("What marks do I need to pass a theory course?",                                 "68903c797f1721754283129"),
    ("How does the make-up exam work and how much can it improve my grade?",          "68903c797f1721754283129"),
    ("What CGPA is needed for promotion to the next year in UG programmes?",          "68903c797f1721754283129"),
    ("Can I pay my fees in monthly EMIs?",                                            "admissions/financial-assistance"),
    ("What scholarships does Presidency University offer?",                           "admissions/scholarships"),
    ("What is Presidency LaunchPad and how does it help student startups?",           "launchpad"),
    ("How many credits do I need to complete B.Tech CSE for the 2023-27 batch?",      "class:official_program_regulation"),
    ("Where can I find previous year question papers for Engineering Chemistry?",     "class:repository_metadata"),
    ("Where is the admissions office on campus?",                                     "class:campus_map|user_photo|campus_tour|user_video"),
    ("How much people traffic is there on campus during the day?",                    "id:people-"),
    ("How many people are there in campus video V2?",                                 "id:people-V2"),
    ("What are the working hours of the Learning and Development department?",        "id:teacher-"),
]

def matches(r, target):
    if target.startswith("id:"):
        return r["id"].startswith(target[3:])
    if target.startswith("class:"):
        return r["source_class"] in target[6:].split("|")
    return target in (r.get("url") or "") or target in (r.get("page_url") or "")

found = 0
for q, target in EVAL:
    t = time.time()
    hits = search(q)
    rank = next((n for n, r in enumerate(hits, 1) if matches(r, target)), None)
    found += rank is not None
    print(f"{'OK  ' if rank else 'MISS'} rank={rank or '-'}  {time.time() - t:.2f}s  {q}")
print(f"\nRight source in top {TOP_K}: {found}/{len(EVAL)}   (plain search; section 9 re-tests with query rewriting)")

MISS rank=-  0.40s  What is the minimum attendance needed to write the end-term exam?
MISS rank=-  0.37s  What marks do I need to pass a theory course?
OK   rank=1  0.41s  How does the make-up exam work and how much can it improve my grade?
OK   rank=1  0.42s  What CGPA is needed for promotion to the next year in UG programmes?
OK   rank=1  0.42s  Can I pay my fees in monthly EMIs?
OK   rank=1  0.40s  What scholarships does Presidency University offer?
OK   rank=4  0.40s  What is Presidency LaunchPad and how does it help student startups?
OK   rank=1  0.48s  How many credits do I need to complete B.Tech CSE for the 2023-27 batch?
OK   rank=1  0.39s  Where can I find previous year question papers for Engineering Chemistry?
OK   rank=2  0.33s  Where is the admissions office on campus?
OK   rank=1  0.31s  How much people traffic is there on campus during the day?
OK   rank=1  0.27s  How many people are there in campus video V2?
OK   rank=1  0.35s  What are the working hours of the Learnin

---
# B. Chatbot
## 6. Load the vision-language model (`MODEL_ID`; default Qwen3-VL-30B-A3B: ~1 min, ~62 GB of GPU memory)

If you changed `MODEL_ID` in step 1, the new model is downloaded here first (one time).

In [8]:
from huggingface_hub import snapshot_download
from transformers import AutoProcessor, AutoModelForImageTextToText

t = time.time()
if not os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    print(f"Downloading {MODEL_ID} -> {MODEL_DIR} (one time)…")
    snapshot_download(MODEL_ID, local_dir=MODEL_DIR)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_DIR, dtype="auto", device_map="cuda:0", attn_implementation="sdpa",
).eval()
processor = AutoProcessor.from_pretrained(MODEL_DIR)
print(f"Loaded in {time.time() - t:.0f}s | GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

Loading weights:   0%|          | 0/882 [00:00<?, ?it/s]

Loaded in 11s | GPU memory used: 65.6 GB


## 7. Model helpers (same as notebook 01)

In [10]:
from PIL import Image

def load_image(path_or_img, max_side=1600):
    img = path_or_img if isinstance(path_or_img, Image.Image) else Image.open(path_or_img)
    img = img.convert("RGB")
    img.thumbnail((max_side, max_side))
    return img

def ask_model(messages, max_new_tokens=512, verbose=True):
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)
    t = time.time()
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             temperature=None, top_p=None, top_k=None)
    new_tokens = out[:, inputs["input_ids"].shape[1]:]
    if verbose:
        n, dt = new_tokens.shape[1], time.time() - t
        print(f"[{inputs['input_ids'].shape[1]} prompt tokens -> {n} new tokens in {dt:.1f}s = {n / dt:.1f} tok/s]")
    return processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()

## 7b. Voice input (Whisper speech-to-text)

Qwen3-VL can't hear audio, so a voice note is first turned into text by **Whisper large-v3-turbo**
(~1.6 GB GPU memory; English, Hindi, Kannada and ~100 other languages, kept in the spoken language).
`av` decodes any format a phone records (m4a, mp3, ogg/opus, wav, webm — even mp4 videos) without ffmpeg on the hub.
The `%pip` line is only needed the first time.

In [11]:
%pip install -q av


[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [12]:
import io, av, numpy as np
from transformers import pipeline

t = time.time()
asr = pipeline("automatic-speech-recognition", model=ASR_MODEL, device=DEVICE, chunk_length_s=30,
               dtype=torch.float16 if DEVICE == "cuda" else torch.float32)
print(f"Whisper loaded in {time.time() - t:.0f}s")

def load_audio(src, sr=16_000):
    """Decode an audio/video file (path or raw bytes) to 16 kHz mono float32 samples."""
    src = io.BytesIO(src) if isinstance(src, (bytes, bytearray, memoryview)) else src
    resampler, chunks = av.AudioResampler(format="flt", layout="mono", rate=sr), []
    with av.open(src) as container:
        for frame in container.decode(audio=0):
            chunks += [f.to_ndarray().reshape(-1) for f in resampler.resample(frame)]
    chunks += [f.to_ndarray().reshape(-1) for f in resampler.resample(None)]   # flush the resampler
    return np.concatenate(chunks) if chunks else np.zeros(0, dtype=np.float32)

def transcribe(audio, verbose=True):
    wav = load_audio(audio)
    t = time.time()
    text = asr({"raw": wav, "sampling_rate": 16_000}, generate_kwargs={"task": "transcribe"})["text"].strip()
    if verbose:
        print(f"[transcribed {len(wav) / 16_000:.0f}s of audio in {time.time() - t:.1f}s]")
    return text

config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.62GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.77k [00:00<?, ?B/s]

...

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

Whisper loaded in 18s


## 8. The RAG pipeline

Edit `SYSTEM_PROMPT` to change the bot's behaviour. `answer_question()` does the work; `chat()` shows the answer nicely.

In [13]:
import re
from IPython.display import display, Markdown

SYSTEM_PROMPT = """You are PU Assist, the student help-desk assistant of Presidency University, Bengaluru.

Answer using the numbered SOURCES given with each question (and any images shown). Rules:
1. University facts must come from the sources. Cite them inline like [1] or [2][4] right after the fact.
2. If the sources do not contain the answer, say clearly that you could not find it in the university documents and suggest which office or page to check. Never invent rules, numbers, dates, fees or names.
3. Rules can differ by programme and admission batch. If that matters and the student has not said, state which document or batch you used and ask for their programme and batch. When sources are different versions of the same rules (for example 2019 vs 2025 regulations), follow the newest one and name it.
4. Official university sources outrank everything else. Student photos, videos, interviews, teacher recordings and external sites are unofficial; say so when you use them. People counts / foot traffic from campus videos are automatic estimates from short recordings: give them as approximate and say when they were recorded (it is a snapshot, not the traffic at every time of day).
5. Fees, dates, timetables and notices change: when you give them, tell the student to confirm with the latest official notice.
6. If the student shares an image (notice, timetable, assignment), use what it says together with the sources, and point out if it conflicts with the sources.
7. For directions, use the campus map image if one is given: describe the route with the block letters and landmarks on the map, and say so if the place is not on the map.
8. If the question is a general study question (for example an assignment problem) rather than about the university, you may answer from your own knowledge: say it is not from university documents and explain the steps so the student learns, rather than only giving the final answer.
9. Be clear and brief: a short direct answer first, then bullet points if needed. Reply in the language the student writes in."""

NAV_WORDS = re.compile(
    r"\b(where is|where are|where's|located|location of|directions?|way to|how (do|can) i (get|go|reach)|"
    r"reach the|which block|which building|which floor|campus map|nearest|navigate)\b", re.I)
MAP_CLASSES = {"campus_map", "campus_tour", "user_photo", "user_video"}

HISTORY, MAX_HISTORY = [], 3        # (question, answer) pairs used for follow-up questions

def reset_chat():
    HISTORY.clear()
    print("New conversation.")

REWRITE_PROMPT = """Turn the student's question into one search query for Presidency University documents (academic regulations, examination rules, circulars, fee and scholarship pages, handbooks, programme curricula).
{history}Student question: {question}

Write ONE search query (max 40 words) that:
- adds missing context from the conversation (programme, batch, topic), if there is any
- uses the formal wording such documents use, e.g. "minimum performance criteria", "End Term Examination", "attendance requirement", "make-up examination", "promotion CGPA", "fee instalment", "scholarship eligibility"
- keeps names, course codes and numbers from the question
Output only the query."""

def rewrite_query(question):
    """Plain questions rarely use the regulations' wording ('pass' vs 'minimum performance criteria'),
    and follow-ups like 'and for MBA?' lack context. Qwen rewrites the question into a search query."""
    history = ""
    if HISTORY:
        convo = "\n".join(f"Student: {q}\nAssistant: {a[:300]}" for q, a in HISTORY[-MAX_HISTORY:])
        history = f"Conversation so far:\n{convo}\n\n"
    msgs = [{"role": "user", "content": [{"type": "text",
             "text": REWRITE_PROMPT.format(history=history, question=question)}]}]
    return ask_model(msgs, max_new_tokens=80, verbose=False).strip().strip('"')

def read_image(img, question):
    """First look at an uploaded image: transcribe it and suggest a search query."""
    msgs = [{"role": "user", "content": [
        {"type": "image", "image": img},
        {"type": "text", "text":
            f'A student uploaded this image and asked: "{question}"\n'
            "1. Transcribe the important text in the image (notice, timetable, question, etc.).\n"
            "2. On the last line write SEARCH: followed by a short query (max 20 words) to look up "
            "the relevant Presidency University rules or information."}]}]
    reply = ask_model(msgs, max_new_tokens=600, verbose=False)
    m = re.search(r"SEARCH:\s*(.+?)\s*$", reply, re.S)
    if m:
        return reply[:m.start()].strip(), m.group(1).strip()
    return reply.strip(), ""

def build_context(hits):
    blocks = []
    for n, r in enumerate(hits, 1):
        kind = CLASS_LABEL.get(r["source_class"], r["source_class"])
        blocks.append(f"[{n}] {source_label(r)} ({kind}) {source_link(r)}\n{r['passage'][:1800]}")
    return "\n\n".join(blocks)

def format_sources(hits, cited):
    lines = []
    for n in (cited or range(1, len(hits) + 1)):
        r = hits[n - 1]
        link = source_link(r)
        where = f"[{source_label(r)}]({link})" if link.startswith("http") else f"{source_label(r)} `{link}`"
        lines.append(f"- [{n}] {where} — {CLASS_LABEL.get(r['source_class'], r['source_class'])}")
    return ("**Sources:**\n" if cited else "**Sources consulted:**\n") + "\n".join(lines)

def answer_question(question="", image=None, audio=None, k=TOP_K):
    timings, t0 = {}, time.time()
    img = load_image(image) if image is not None else None
    question = (question or "").strip()

    # 0. voice: a voice note alone IS the question; with a typed question it is extra context
    #    (e.g. a recorded announcement + "what does this mean for me?")
    audio_text = transcribe(audio, verbose=False) if audio is not None else ""
    voice_question = bool(audio_text) and not question
    if voice_question:
        question = audio_text
    if not question:
        raise ValueError("Type a question or add a voice note.")
    if audio is not None:
        timings["listen"] = time.time() - t0

    # 1. understand: rewrite the question in the documents' wording, read the image
    queries, reading = [rewrite_query(question)], ""
    if img is not None:
        reading, img_query = read_image(img, question)
        queries.append(img_query or reading[:300])
    if audio_text and not voice_question:
        queries.append(audio_text[:300])
    # rerank against the bare question unless it needs the history, image or audio to make sense
    needs_context = HISTORY or img is not None or (audio_text and not voice_question)
    rerank_query = "\n".join([question] + queries) if needs_context else question
    timings["understand"] = time.time() - t0 - timings.get("listen", 0)

    # 2. retrieve
    t = time.time()
    hits = search(rerank_query, k=k, extra_queries=queries)
    timings["search"] = time.time() - t

    # 3. answer with sources (+ uploaded image, + campus map for directions)
    use_map = os.path.exists(CAMPUS_MAP) and (
        bool(NAV_WORDS.search(question)) or any(h["source_class"] in MAP_CLASSES for h in hits[:3]))
    content, notes = [], []
    if img is not None:
        content.append({"type": "image", "image": img})
        notes.append("IMAGE 1 is the student's upload. Text read from it:\n" + reading[:2000])
    if use_map:
        content.append({"type": "image", "image": load_image(CAMPUS_MAP, max_side=2000)})
        notes.append(f"{'IMAGE 2' if img is not None else 'The image'} is the official campus map sketch.")
    if audio_text and not voice_question:
        notes.append("The student also shared an audio recording. Automatic transcript (may contain errors):\n"
                     + audio_text[:3000])
    content.append({"type": "text", "text":
        f"SOURCES:\n{build_context(hits)}\n\n" + "".join(n + "\n\n" for n in notes) +
        f"STUDENT QUESTION: {question}"})

    messages = [{"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]}]
    for q, a in HISTORY[-MAX_HISTORY:]:
        messages += [{"role": "user", "content": [{"type": "text", "text": q}]},
                     {"role": "assistant", "content": [{"type": "text", "text": re.sub(r"\s*\[\d+\]", "", a)}]}]
    messages.append({"role": "user", "content": content})

    t = time.time()
    answer = ask_model(messages, max_new_tokens=700, verbose=False)
    timings["answer"] = time.time() - t
    HISTORY.append((question, answer))

    cited = sorted({int(n) for n in re.findall(r"\[(\d+)\]", answer) if 0 < int(n) <= len(hits)})
    timing_note = " · ".join(f"{k} {v:.1f}s" for k, v in timings.items()) + (" · used campus map" if use_map else "")
    heard = f"🎤 *Heard:* “{question}”\n\n" if voice_question else ""
    markdown = f"{heard}{answer}\n\n{format_sources(hits, cited)}\n\n<sub>⏱ {timing_note}</sub>"
    return {"question": question, "answer": answer, "markdown": markdown, "hits": hits, "cited": cited,
            "queries": queries, "image_text": reading, "audio_text": audio_text,
            "used_map": use_map, "timings": timings}

def chat(question="", image=None, audio=None):
    result = answer_question(question, image, audio)
    display(Markdown(result["markdown"]))
    return result

---
# C. Try it
## 9. Retrieval check again, now with query rewriting

Same 13 questions as step 5, but each is first rewritten by Qwen3-VL. Compare the score with step 5.

In [14]:
found = 0
for q, target in EVAL:
    HISTORY.clear()
    t = time.time()
    rq = rewrite_query(q)
    hits = search(q, extra_queries=[rq])
    rank = next((n for n, r in enumerate(hits, 1) if matches(r, target)), None)
    found += rank is not None
    print(f"{'OK  ' if rank else 'MISS'} rank={rank or '-'}  {time.time() - t:.1f}s  {q}\n            rewritten: {rq}")
print(f"\nRight source in top {TOP_K}: {found}/{len(EVAL)}   (with query rewriting)")

MISS rank=-  2.1s  What is the minimum attendance needed to write the end-term exam?
            rewritten: Minimum attendance requirement to write End Term Examination for [Programme Name], [Batch Year], as per Presidency University Academic Regulations and Examination Rules.
MISS rank=-  2.3s  What marks do I need to pass a theory course?
            rewritten: What is the minimum performance criteria for passing a theory course in the B.Sc. Physics programme, Batch 2022, as per the Academic Regulations and Examination Rules of Presidency University?
OK   rank=1  2.5s  How does the make-up exam work and how much can it improve my grade?
            rewritten: Make-up examination policy, grade improvement rules, and minimum performance criteria for B.Sc. (Hons) in Computer Science, Batch 2022, as per Presidency University Academic Regulations and Examination Rules.
OK   rank=2  2.1s  What CGPA is needed for promotion to the next year in UG programmes?
            rewritten: Minimum CG

## 10. A text question

In [15]:
reset_chat()
r = chat("What is the minimum attendance needed to write the end-term exam?")

New conversation.


The minimum attendance required to write the end-term exam is **75%**.

This is stated in the **PU Factsheet 2025-26** [2] and also confirmed in multiple circulars, including **Circular (PU202383)** and **Circular (PU202396)** for the Academic Year 2023-24, which specify that a student must have a minimum of 75% attendance in the classes actually conducted for the courses they are registered in to be eligible to write the final examination [3].

> **Note:** This rule applies to all programs and is consistent across the latest regulations. Always confirm with the latest official notice or the Controller of Examinations for any updates.

**Sources:**
- [2] [PU Factsheet 2025-26](https://presidencyuniversity.in/uploads/images/6791f7156eec81737619221.pdf) — Official document
- [3] [Presidency University](https://presidencyuniversity.in/school/school-of-management/circulars) — Official website

<sub>⏱ understand 1.2s · search 0.6s · answer 6.4s</sub>

## 11. A follow-up (uses the chat history)

In [16]:
r = chat("What if I missed classes because I was in hospital?")
print("Search queries used:", r["queries"])

If you missed classes because you were in the hospital, you may be eligible for **relaxation in the attendance requirement**, but **you must meet specific conditions**.

### ✅ Minimum Attendance Requirement:
- The general minimum attendance required to write the end-term exam is **75%** of the classes actually conducted [1][2][3][4][5][6].

### ✅ Exception for Hospitalization:
- If you were hospitalized due to **serious medical exigencies**, including **hospitalization, trauma, or contagious disease**, you may be granted **relaxation** in attendance.
- The **minimum attendance requirement can be reduced to 65%** in such cases, **but only if**:
  - You **inform your HOD/Dean timely** (on or before the last date of the end-term exams) [2][5].
  - You **submit authentic medical documents**, such as:
    - Medical certificates
    - Hospital discharge summary
    - Medical fitness report
    - Prescriptions
    - All duly attested by a registered medical officer [2][5].

> ⚠️ **Important**: This relaxation is **not automatic**. It is granted **only on the recommendation of the Dean** and **approval by the Vice Chancellor**. You must apply for it and provide all required documentation.

### ❌ What is not allowed:
- You **cannot** be exempted from attendance requirements for **non-medical reasons**.
- You **cannot** use this provision if you **fail to produce valid medical proof**.

### ✅ What to do:
1. **Inform your HOD/Dean immediately** after your hospitalization.
2. **Submit all medical documents** as soon as possible.
3. **Apply for Make-Up Examination** if you missed the end-term exam due to hospitalization (if eligible).

> 📌 **Note**: The latest regulations (Academic Regulation 2025 and LL.M 2025-26/2026-27) confirm that **65% is the lowest possible attendance** allowed under special circumstances, but **75% is the standard requirement**.

For the most accurate guidance, contact the **Controller of Examinations** or your **Dean/HOD** with your medical records.

**Sources:**
- [1] [PU Factsheet 2025-26](https://presidencyuniversity.in/uploads/images/6791f7156eec81737619221.pdf) — Official document
- [2] [Academic Regulation 2025](https://presidencyuniversity.in/uploads/images/68903c797f1721754283129.pdf) — Official document
- [3] [LL.M 2025-26](https://presidencyuniversity.in/uploads/images/685b9d0e3168c1750834446.pdf) — Official document
- [4] [LL.M 2026-27](https://presidencyuniversity.in/uploads/images/6a61f6587f46f1784804952.pdf) — Official document
- [5] [Academic Regulation 2025](https://presidencyuniversity.in/uploads/images/68903c797f1721754283129.pdf) — Official document
- [6] [Academic Regulations 2020](https://presidencyuniversity.in/assets/PDF/Academic-Regulations-2020.pdf) — Official document

<sub>⏱ understand 2.0s · search 0.6s · answer 19.0s</sub>

Search queries used: ['Search query: "Presidency University attendance requirement for End Term Examination, minimum 75%, exceptions for medical absence, hospitalization, make-up examination, and academic regulations for students with medical certificates, 2025-26.']


## 12. An image question

Upload a notice / timetable / assignment photo with the ⬆ button and put its name below.

In [17]:
reset_chat()
IMAGE_PATH = "notice.jpg"     # <- your uploaded image

if os.path.exists(IMAGE_PATH):
    r = chat("What does this mean for me as a student?", image=IMAGE_PATH)
    print("Text the model read from the image:\n", r["image_text"][:800])
else:
    print(f"Upload an image first: {IMAGE_PATH} not found")

New conversation.
Upload an image first: notice.jpg not found


## 13. A voice question

Record a voice note on your phone (e.g. *"What is the minimum attendance for the end-term exam?"*, in English,
Hindi or Kannada), upload it with ⬆ and put its name below. Add a typed question too if the recording is
something to explain (e.g. a recorded announcement + "What does this mean for me?").

In [18]:
reset_chat()
AUDIO_PATH = "voice_question.m4a"     # <- your uploaded voice note (m4a, mp3, ogg, opus, wav, webm)

if os.path.exists(AUDIO_PATH):
    print("Whisper heard:", transcribe(AUDIO_PATH))
    r = chat(audio=AUDIO_PATH)
else:
    print(f"Upload a voice note first: {AUDIO_PATH} not found")

New conversation.
Upload a voice note first: voice_question.m4a not found


## 14. A directions question (adds the campus map automatically)

In [19]:
reset_chat()
r = chat("Where is the Registrar's office?")

New conversation.


Based on the provided campus map sketch, the Registrar's office is located in the **Centria Admin** building.

This building is situated in the central part of the campus, near the main roundabout and the Canopy. The map shows the "Registrar Office" specifically labeled within the Centria Admin block.

The sources confirm that the Office of the Registrar is a key administrative unit at the university [4][5]. While the provided photos [1][2][3] show the office's location and counter, the campus map [image] provides the definitive location on the physical campus.

**Sources:**
- [1] office of registration.jpg `dataset_sources/user_media/compressed/Universitypics/office of registration.jpg` — Student photo caption (unofficial)
- [2] IMG20261029103647.jpg `dataset_sources/user_media/compressed/Universitypics/IMG20261029103647.jpg` — Student photo caption (unofficial)
- [3] IMG20261029103649.jpg `dataset_sources/user_media/compressed/Universitypics/IMG20261029103649.jpg` — Student photo caption (unofficial)
- [4] [Registrar | Presidency University](https://presidencyuniversity.in/about-us/leadership/registrar) — Official website
- [5] [Registrar | Presidency University](https://presidencyuniversity.in/about-us/leadership/registrar) — Official website

<sub>⏱ understand 1.6s · search 0.7s · answer 59.8s · used campus map</sub>

## 15. Answer check over the test questions (~3–5 min)

Read each answer and check: is it correct, are the [n] citations the right documents, does it admit when it doesn't know?

In [20]:
for q, _ in EVAL:
    reset_chat()
    display(Markdown(f"---\n### Q: {q}"))
    chat(q)

New conversation.


---
### Q: What is the minimum attendance needed to write the end-term exam?

The minimum attendance required to write the end-term exam is **75%**.

This is stated in the **PU Factsheet 2025-26** [2] and also confirmed in multiple circulars, including **Circular (PU202383)** and **Circular (PU202396)** for the Academic Year 2023-24, which specify that a student must have a minimum of 75% attendance in the classes actually conducted for the courses they are registered in to be eligible to write the final examination [3].

> **Note:** This rule applies to all programs and is consistent across the latest regulations. Always confirm with the latest official notice or the Controller of Examinations for any updates.

**Sources:**
- [2] [PU Factsheet 2025-26](https://presidencyuniversity.in/uploads/images/6791f7156eec81737619221.pdf) — Official document
- [3] [Presidency University](https://presidencyuniversity.in/school/school-of-management/circulars) — Official website

<sub>⏱ understand 1.2s · search 0.6s · answer 6.2s</sub>

New conversation.


---
### Q: What marks do I need to pass a theory course?

...

---
### Q: How many people are there in campus video V2?

Based on the provided sources, the campus video V2 shows moderate foot traffic.

At most, 15 people are visible at the same time (specifically at 1:45 into the video). On average, about 4.6 people are visible throughout the 142-second video. The total number of different people cannot be reliably counted from the video.

**Source:** [1] Campus video V2 - people count (Student campus video (unofficial, auto-transcribed)) keshavs_input/V2.mp4

**Sources:**
- [1] Campus video V2 - people count `keshavs_input/V2.mp4` — Student campus video (unofficial, auto-transcribed)

<sub>⏱ understand 1.8s · search 0.5s · answer 58.5s · used campus map</sub>

New conversation.


---
### Q: What are the working hours of the Learning and Development department?

The working hours of the Learning and Development department are **Monday to Friday, 8:50 AM to 4:15 PM**, and the department also works on Saturdays [1]. Faculty members teach approximately 18 to 20 hours of classes per week [1].

**Sources:**
- [1] Teacher recording - Learning and Development department (summary) `keshavs_input/Teacher.m4a` — Teacher recording (auto-transcribed, not an official document)

<sub>⏱ understand 1.4s · search 0.5s · answer 2.3s</sub>

---
# D. Chat box inside the notebook (step 16)

Type a question and/or add a voice note, optionally add an image, press **Ask**.
Works on this hub (uses `ipywidgets`, no web link needed). Live microphone recording isn't possible
inside a notebook — record a voice note on your phone and upload it.
One question at a time; the conversation history is used for follow-ups until you press **New conversation**.

In [21]:
import io
import ipywidgets as widgets

question_box  = widgets.Textarea(placeholder="Ask about exams, attendance, fees, scholarships, campus…",
                                 layout=widgets.Layout(width="100%", height="80px"))
image_upload  = widgets.FileUpload(accept="image/*", multiple=False, description="Add image")
image_preview = widgets.Image(layout=widgets.Layout(max_width="240px", display="none"))
audio_upload  = widgets.FileUpload(accept="audio/*,.m4a,.mp3,.wav,.ogg,.opus,.webm", multiple=False,
                                   description="Add voice")
ask_button    = widgets.Button(description="Ask", button_style="primary", icon="paper-plane")
new_button    = widgets.Button(description="New conversation", icon="refresh")
status        = widgets.HTML()
conversation  = widgets.Output(layout=widgets.Layout(border="1px solid #ccc", padding="8px",
                                                     max_height="600px", overflow="auto"))

def uploaded_image():
    if not image_upload.value:
        return None
    return Image.open(io.BytesIO(bytes(image_upload.value[0]["content"])))

def on_upload(_):
    if image_upload.value:
        f = image_upload.value[0]
        image_preview.format = (f.get("type") or "image/png").split("/")[-1]
        image_preview.value = bytes(f["content"])
        image_preview.layout.display = ""
    else:
        image_preview.layout.display = "none"

def uploaded_audio():
    return bytes(audio_upload.value[0]["content"]) if audio_upload.value else None

def on_audio(_):
    if audio_upload.value:
        status.value = f"<i>🎤 {audio_upload.value[0]['name']} added</i>"

def on_ask(_):
    question, img, audio = question_box.value.strip(), uploaded_image(), uploaded_audio()
    if not question and audio is None:
        status.value = "<i>Type a question or add a voice note.</i>"
        return
    ask_button.disabled, status.value = True, "<i>Thinking… (10–40 s)</i>"
    try:
        result = answer_question(question, img, audio)
        extras = [label for label, used in (("with image", img is not None), ("voice", audio is not None)) if used]
        with conversation:
            display(Markdown(f"**You:** {result['question']}" + (f" *({', '.join(extras)})*" if extras else "")))
            display(Markdown(result["markdown"] + "\n\n---"))
        question_box.value, image_upload.value, audio_upload.value, status.value = "", (), (), ""
    except Exception as e:
        status.value = f"<b style='color:#c00'>Error:</b> {type(e).__name__}: {e}"
    finally:
        ask_button.disabled = False

def on_new(_):
    HISTORY.clear()
    conversation.clear_output()
    status.value = "<i>New conversation.</i>"

image_upload.observe(on_upload, names="value")
audio_upload.observe(on_audio, names="value")
ask_button.on_click(on_ask)
new_button.on_click(on_new)

display(widgets.VBox([
    widgets.HTML("<h3>PU Assist – Presidency University help desk</h3>"),
    conversation,
    question_box,
    widgets.HBox([image_upload, audio_upload, ask_button, new_button, status]),
    image_preview,
]))

---
# E. Optional: Gradio web page (step 17, only on hubs with `jupyter-server-proxy`)

Starts a small Gradio app behind JupyterHub's proxy and prints its link.
- If the link shows **404 / page not found**, this hub doesn't have `jupyter-server-proxy`; use the chat box in D.
- One shared chat history, one question at a time — fine for a demo, not for many students
  (the real web app comes later with a proper backend).
- Stop it with `demo.close()`.

In [22]:
import gradio as gr, socket

try:
    demo.close()          # stop the server from a previous run of this cell
except NameError:
    pass

def ui_answer(question, image, audio_path):
    if not (question or "").strip() and not audio_path:
        return "Please type a question or record/upload a voice note."
    return answer_question(question, image, audio_path)["markdown"]

def ui_reset():
    HISTORY.clear()
    return ""

with gr.Blocks(title="PU Assist") as demo:
    gr.Markdown("## PU Assist – Presidency University help desk")
    with gr.Row():
        with gr.Column(scale=1):
            img_in = gr.Image(type="pil", label="Optional image (notice, timetable, assignment)")
            audio_in = gr.Audio(sources=["microphone", "upload"], type="filepath", label="Optional voice question")
        with gr.Column(scale=2):
            q_in = gr.Textbox(label="Your question", lines=3)
            ask_btn = gr.Button("Ask", variant="primary")
            new_btn = gr.Button("New conversation")
    out = gr.Markdown()
    ask_btn.click(ui_answer, [q_in, img_in, audio_in], out)
    q_in.submit(ui_answer, [q_in, img_in, audio_in], out)
    new_btn.click(ui_reset, None, out)

with socket.socket() as s:            # let the OS pick a free port (7860 may be taken)
    s.bind(("127.0.0.1", 0))
    PORT = s.getsockname()[1]

prefix = os.environ.get("JUPYTERHUB_SERVICE_PREFIX", "/")
demo.launch(server_name="127.0.0.1", server_port=PORT, root_path=f"{prefix}proxy/{PORT}",
            inline=False, prevent_thread_lock=True)
print("Open this in a new tab:", f"https://compute.data-lab.site{prefix}proxy/{PORT}/")

* Running on local URL:  http://127.0.0.1:49323
* To create a public link, set `share=True` in `launch()`.
Open this in a new tab: https://compute.data-lab.site/user/<user>/proxy/49323/
